Mi logica planteada:
17. Pac-Man mini: Construye un agente para recoger puntos en un mapa pequeño. Explica cómo explora zonas no visitadas y cómo explota caminos con más puntos y menos riesgo.

Entorno = Matriz de tiles 
El entorno sera uno definido pero que tiene varios caminos siendo que la matriz sera descrita de la siguiente manera:
	1 = punto
	0 = espacio vacio
	2 = borde o pared
	3 = el pacman 
Mapa1 = 
[[2,2,2,2,2,2],
 [2,1,1,1,1,2],
 [2,1,1,3,1,2],
 [2,1,1,1,1,2],
 [2,1,1,1,1,2],
 [2,2,2,2,2,1]		
]

El agente tendra que aprender este mapa, podemos enseñarle cada vez al agente en un entorno diferente gracias a la clase juego y su función de valor sera generada y diferente dando el mejor camino posible para cada mapa.
Ahora también cada que el agente se mueve de estado y hace un movimiento valido debe tener un método update() que cambie el uno que había en la matriz por un 0 que significa que no hay puntos ahi y asignar al agente un punto por haberlo recogido
Aquí me surge una duda, como puedo calcular la función de valor de mi agente? si en el ejemplo de tic tac toe era de ganar en toda una partida para el caso de pacman tendría que ser cunado termina toda una partida y logra recoger o cada vez que el agente come un punto? primero probare con el agente que come un punto.
Entonces esta clase se llamara Mapa()
El mapa debe tener como metodo un metodo que vea si aun hay puntos y si ya no hay puntos que saque true (puede llamarase valid_moves())
Mapa() Tiene de metodos su state que sera cargado en el constructor y el usuario puede definirlo como en el ejemlo que puse en Mapa1 al recibir esto en el constructor entonces ese se carga en el objeto


Una clase Juego que orquestara el entrenamiento del agente, cuando en su mapa no hayan mas puntos para recoger entonces el juego termina y el pac man gana.
Se puede colocar al metodo como parametro la cantidad de partidas.

Agente = Explora o explota.
El agente tiene como objetivo recoger todos los puntos en el mapa y en base a eso buscar el mejor camino para recoger con la mayor cantidad de recompensa.
Incluido en el agente:
Política = Decidir si seguir explotando o explorar, esto permitirá al agente encontrar caminos mas variados mediante un parametro llamado prob_exp el cual es de 0.1 a 1 e indica una probabilidad de exploración si el margen es debajo de este valor aleatorio que salga entonces explorara a un movimiento aleatorio de sus alrededores.
Como método move() solo puede moverse en sus espacios cercanos y debe ser o un 1 o un 0 en el mapa los bordes del mapa serán pues números 2 y yo puedo definir mi matriz de tiles asi que eso me facilitara las cosas.
Función de valor = Sera un diccionario y contendra todos los caminos posibles desde un estado que sumados llevaran a la mayor cantidad de recompensa final. (aquí aplicare la formula de actualización temporal de modo que se actualizara en cada partida.) 
Recompensa = Cada que recoge un punto es recompensado con 1 punto, su objetivo final es recoger todos los puntos repartidos por el mapa.
Cada vez que consume un movimiento sin comer nada es castigado con -1 punto o sea cada vez que su movimiento se dirija a un espacio ya recogido como podria ser un 0. Su objetivo final sera conseguir las mejores funciones valor desde tales posiciones


In [1]:
import numpy as np
import random

np.random.seed(42)
random.seed(42)

Mapa1 = [
    [2, 2, 2, 2, 2, 2],
    [2, 3, 1, 1, 1, 2],
    [2, 1, 1, 1, 1, 2],
    [2, 1, 1, 1, 1, 2],
    [2, 1, 1, 1, 1, 2],
    [2, 2, 2, 2, 2, 2]
]

class Mapa():
    def __init__(self, state):
        self.initial_state = np.array(state)
        self.reset()

    def reset(self):
        self.state = self.initial_state.copy()
        self.points = 0

        for i in range(self.state.shape[0]):
            for j in range(self.state.shape[1]):
                if self.state[i, j] == 3:
                    self.pacman = (i, j)

    def get_state(self):
        return str(self.state.reshape(self.state.shape[0] * self.state.shape[1]))

    def has_points(self):
        return 1 in self.state

    def valid_moves(self):
        moves = []
        i, j = self.pacman

        posibles = [
            (i - 1, j),
            (i + 1, j),
            (i, j - 1),
            (i, j + 1)
        ]

        for move in posibles:
            fila, columna = move
            if self.state[fila, columna] == 0 or self.state[fila, columna] == 1:
                moves.append(move)

        return moves

    def update(self, move):
        i, j = self.pacman
        fila, columna = move

        if self.state[fila, columna] == 1:
            reward = 1
            self.points += 1
        else:
            reward = -1

        self.state[i, j] = 0
        self.state[fila, columna] = 3
        self.pacman = move

        return reward

    def show(self):
        print(self.state)


In [2]:
mapa = Mapa(Mapa1)
mapa.show()
mapa.valid_moves()

[[2 2 2 2 2 2]
 [2 3 1 1 1 2]
 [2 1 1 1 1 2]
 [2 1 1 1 1 2]
 [2 1 1 1 1 2]
 [2 2 2 2 2 2]]


[(2, 1), (1, 2)]

In [3]:
class Agente():
    def __init__(self, alpha=0.5, gamma=0.9, prob_exp=0.3):
        self.value_function = {}
        self.alpha = alpha
        self.gamma = gamma
        self.prob_exp = prob_exp
        self.positions = []

    def reset(self):
        self.positions = []

    def value(self, state):
        if state not in self.value_function:
            self.value_function[state] = 0
        return self.value_function[state]

    def move(self, mapa):
        moves = mapa.valid_moves()

        if len(moves) == 0:
            return None

        if random.random() < self.prob_exp:
            move = moves[np.random.choice(len(moves))]
        else:
            values = []

            for move in moves:
                old_i, old_j = mapa.pacman
                new_i, new_j = move

                next_state = mapa.state.copy()
                next_state[old_i, old_j] = 0
                next_state[new_i, new_j] = 3
                next_state = str(next_state.reshape(next_state.shape[0] * next_state.shape[1]))

                values.append(self.value(next_state))

            max_value = max(values)
            best_moves = [moves[i] for i in range(len(moves)) if values[i] == max_value]
            move = best_moves[np.random.choice(len(best_moves))]

        state = mapa.get_state()
        reward = mapa.update(move)
        next_state = mapa.get_state()

        self.positions.append(state)
        self.learn(state, reward, next_state)

        return reward

    def learn(self, state, reward, next_state):
        value = self.value(state)
        next_value = self.value(next_state)
        self.value_function[state] = value + self.alpha * (reward + self.gamma * next_value - value)


In [4]:
class Juego():
    def __init__(self, agente, mapa):
        self.agente = agente
        self.mapa = mapa

    def entrenar(self, partidas=1000, max_steps=100):
        rewards = []
        points = []

        for _ in range(partidas):
            self.mapa.reset()
            self.agente.reset()

            total_reward = 0
            steps = 0

            while self.mapa.has_points() and steps < max_steps:
                reward = self.agente.move(self.mapa)
                total_reward += reward
                steps += 1

            rewards.append(total_reward)
            points.append(self.mapa.points)

        return rewards, points


In [5]:
mapa = Mapa(Mapa1)
agente = Agente(alpha=0.5, gamma=0.9, prob_exp=0.3)
juego = Juego(agente, mapa)

rewards, points = juego.entrenar(partidas=500, max_steps=60)
